# Weather vs Humans: the part of Mumbai's smoke we chose
Google Mumbai Data Challenge, Most Creative entry, by Aaryan Singh.

A weather-only model predicts PM2.5. Actual minus expected is the human-made extra. Upload the challenge files `Train.csv`, `History_Mumbai_2018_2020.csv.gz` and `History_Mumbai_2024.csv.gz` to this Colab session first.

In [ ]:
!pip -q install lightgbm
import pandas as pd, numpy as np, lightgbm as lgb, requests
from sklearn.metrics import r2_score

In [ ]:
import json,pandas as pd,numpy as np,lightgbm as lgb
D='./'
# weather
w1=pd.DataFrame(requests.get('https://archive-api.open-meteo.com/v1/archive',params=dict(latitude=19.086115,longitude=72.85291,start_date='2018-01-01',end_date='2026-10-31',hourly='wind_speed_10m,wind_direction_10m,boundary_layer_height,precipitation,temperature_2m,relative_humidity_2m',models='era5',timezone='Asia/Kolkata')).json()['hourly'])
for f in ['blh24a.json','blh24.json']:
    b=pd.DataFrame(json.load(open(f))['hourly']).rename(columns={'boundary_layer_height':'b2'})
    w1=w1.merge(b,on='time',how='left',suffixes=('','_x'))
    w1['boundary_layer_height']=w1.boundary_layer_height.fillna(w1.b2); w1=w1.drop(columns=['b2'])
w2=pd.read_csv(D+'wx_hist.csv')
w=pd.concat([w1,w2]).drop_duplicates('time',keep='last'); w['ts']=pd.to_datetime(w.time); w=w.set_index('ts').sort_index().drop(columns='time')
w=w.asfreq('h'); print('wx NaN',w.isna().sum().to_dict())
w['ws6']=w.wind_speed_10m.rolling(6,1).mean(); w['ws24']=w.wind_speed_10m.rolling(24,1).mean(); w['ws48']=w.wind_speed_10m.rolling(48,1).mean()
w['blh6']=w.boundary_layer_height.rolling(6,1).mean(); w['blh24']=w.boundary_layer_height.rolling(24,1).mean(); w['blhmax24']=w.boundary_layer_height.rolling(24,1).max()
w['pr24']=w.precipitation.rolling(24,1).sum(); w['pr72']=w.precipitation.rolling(72,1).sum(); w['pr240']=w.precipitation.rolling(240,1).sum()
w['wd_s']=np.sin(np.radians(w.wind_direction_10m)); w['wd_c']=np.cos(np.radians(w.wind_direction_10m))
w['wd6s']=w.wd_s.rolling(6,1).mean(); w['wd6c']=w.wd_c.rolling(6,1).mean(); w['wd24s']=w.wd_s.rolling(24,1).mean(); w['wd24c']=w.wd_c.rolling(24,1).mean()
# land breeze share (wind from east/NE quadrant 0-150 deg) last 24h
w['land24']=((w.wind_direction_10m>=0)&(w.wind_direction_10m<=150)).astype(float).rolling(24,1).mean()
w['rh24']=w.relative_humidity_2m.rolling(24,1).mean(); w['t24']=w.temperature_2m.rolling(24,1).mean()
w=w.reset_index()
# pm
a=pd.read_csv(D+'History_Mumbai_2018_2020.csv.gz'); a=pd.DataFrame({'ts':pd.to_datetime(a.Datetime),'st':a.StationName.str.split(',').str[0],'pm25':a['PM2.5']})
b=pd.read_csv(D+'History_Mumbai_2024.csv.gz'); b=pd.DataFrame({'ts':pd.to_datetime(b.timestamp_ist.astype(str).str[:19]).dt.floor('h'),'st':b.station_name.str.split(',').str[0],'pm25':b.value})
b=b.groupby(['ts','st'],as_index=False).pm25.mean()
t=pd.read_csv(D+'Train.csv'); t=pd.DataFrame({'ts':pd.to_datetime(t.timestamp_ist.str[:19]),'st':t.station_name.str.split(',').str[0],'pm25':t.pm25})
t['st']=t.st+' (OpenAQ)'
p=pd.concat([a.assign(src='cpcb18'),b.assign(src='xkdr24'),t.assign(src='zindi')]).dropna()
p=p[(p.pm25>0)&(p.pm25<800)]
print(p.groupby('src').agg(n=('pm25','size'),st=('st','nunique'),t0=('ts','min'),t1=('ts','max')))
d=p.merge(w,on='ts',how='inner')
d.to_parquet('all.parquet') if False else d.to_pickle('all.pkl')
print(len(d))


Note: the cell above originally merged extra boundary-layer files; in Colab the single ERA5 call replaces them. If the merge lines error, delete the `for f in [...]` loop and the `w2`/concat lines and use `w=w1`.

In [ ]:
import pandas as pd,numpy as np,lightgbm as lgb
from sklearn.metrics import r2_score
d=pd.read_pickle('all.pkl')
d['hour']=d.ts.dt.hour; d['dow']=d.ts.dt.day_of_week; d['stc']=d.st.astype('category')
d['yr']=d.ts.dt.year
F=['stc','hour','dow','wind_speed_10m','wd_s','wd_c','boundary_layer_height','precipitation','temperature_2m','relative_humidity_2m','ws6','ws24','ws48','blh6','blh24','blhmax24','pr24','pr72','pr240','wd6s','wd6c','wd24s','wd24c','land24','rh24','t24']
y=np.log1p(d.pm25); d['exp']=np.nan
for yr in sorted(d.yr.unique()):
    tr=d.yr!=yr
    m=lgb.LGBMRegressor(objective='huber',alpha=0.4,n_estimators=800,learning_rate=0.03,num_leaves=63,min_child_samples=100,subsample=0.7,subsample_freq=1,colsample_bytree=0.7,verbose=-1)
    m.fit(d.loc[tr,F],y[tr]); d.loc[~tr,'exp']=np.expm1(m.predict(d.loc[~tr,F]))
d['date']=d.ts.dt.date
day=d.groupby(['src','st','date']).agg(actual=('pm25','mean'),expected=('exp','mean'),n=('pm25','size')).reset_index()
day=day[day.n>=18]
city=day.groupby('date').agg(actual=('actual','mean'),expected=('expected','mean'),k=('st','size')).reset_index()
city=city[city.k>=3]; city['extra']=city.actual-city.expected
print('daily st R2',r2_score(day.actual,day.expected),'city R2',r2_score(city.actual,city.expected))
for y_ in [2018,2019,2024,2025,2026]:
    c=city[pd.to_datetime(city.date).dt.year==y_]; 
    if len(c): print(y_,'R2',round(r2_score(c.actual,c.expected),2),'n',len(c))
city['m']=pd.to_datetime(city.date).dt.to_period('M')
print(city.groupby('m')[['actual','expected','extra']].mean().round(1).to_string())
print(city.sort_values('extra',ascending=False).head(30).to_string())
d[['ts','src','st','pm25','exp']].to_pickle('hourly2.pkl'); day.to_csv('daily2.csv',index=False); city.to_csv('city2.csv',index=False)


In [ ]:
import pandas as pd,numpy as np
h=pd.read_pickle('hourly2.pkl'); h['date']=h.ts.dt.normalize()
day=h.groupby(['src','st','date']).agg(actual=('pm25','mean'),exp=('exp','mean'),n=('pm25','size')).reset_index(); day=day[day.n>=18]; day['res']=day.actual-day.exp
city=day.groupby('date').agg(actual=('actual','mean'),exp=('exp','mean'),res=('res','mean'),k=('st','size'))
city=city[city.k>=3].asfreq('D')
def diff(t,L=12):
    b=city.loc[t-pd.Timedelta(days=L):t-pd.Timedelta(days=1)]; a=city.loc[t+pd.Timedelta(days=1):t+pd.Timedelta(days=L)]
    if b.res.count()<8 or a.res.count()<8: return None
    return dict(act=a.actual.mean()-b.actual.mean(), exp=a.exp.mean()-b.exp.mean(), res=a.res.mean()-b.res.mean(),
                b_act=b.actual.mean(),a_act=a.actual.mean(),b_exp=b.exp.mean(),a_exp=a.exp.mean())
g=diff(pd.Timestamp('2025-11-27')); print({k:round(v,1) for k,v in g.items()})
pl=[]
for t in city.index:
    if t.month in (10,11,12,1,2) and abs((t-pd.Timestamp('2025-11-27')).days)>30:
        r=diff(t)
        if r: pl.append(r['res'])
pl=np.array(pl); print('placebo n',len(pl),'share <= grap',(pl<=g['res']).mean(),'pctiles',np.percentile(pl,[1,5,50,95]).round(1))
print(city.loc['2025-11-10':'2025-12-20',['actual','exp','res']].round(1).to_string())


In [ ]:
import pandas as pd,numpy as np,json
h=pd.read_pickle('hourly2.pkl'); h['res']=h.pm25-h.exp; h['date']=h.ts.dt.normalize()
day=h.groupby(['src','st','date']).agg(actual=('pm25','mean'),exp=('exp','mean'),n=('pm25','size')).reset_index(); day=day[day.n>=18]; day['res']=day.actual-day.exp
city=day.groupby('date').agg(actual=('actual','mean'),exp=('exp','mean'),res=('res','mean'),k=('st','size')).reset_index(); city=city[city.k>=3].set_index('date').asfreq('D')
# local baseline: median residual of days -21..-8 and +8..+21
def local(dt,w=2):
    dt=pd.Timestamp(dt); win=city.loc[dt-pd.Timedelta(days=w):dt+pd.Timedelta(days=w)]
    base=pd.concat([city.loc[dt-pd.Timedelta(days=21):dt-pd.Timedelta(days=8)],city.loc[dt+pd.Timedelta(days=8):dt+pd.Timedelta(days=21)]]).res.median()
    return win,base
out={}
for name,dt in {'2018':'2018-11-07','2019':'2019-10-27','2024':'2024-11-01','2025':'2025-10-21'}.items():
    win,base=local(dt); win=win.dropna()
    if len(win)==0: print(name,"no data"); continue
    extra=(win.res-base); 
    out[name]=dict(days=len(win),actual_mean=round(win.actual.mean(),1),weather_expected=round(win.exp.mean(),1),extra_mean=round(extra.mean(),1),peak_day=str(win.actual.idxmax().date()),peak=round(win.actual.max(),1),extra_sum_ugday=round(extra.clip(lower=0).sum(),1),cig=round(extra.clip(lower=0).sum()/22,1))
    print(name,out[name]); print(win[['actual','exp','res']].round(1).assign(extra=extra.round(1)).to_string())
# placebo: distribution of 5-day window extras over all non-Diwali Oct-Nov days
